# Phase 9 — Tests de résistance

## Pourquoi cette phase est importante

Les phases 7 et 8 décrivent ce qui s'est passé sur la période. Un test de résistance pose une question différente : que deviendraient la performance cumulée, le drawdown maximal, la volatilité et le ratio de Sharpe si l'on imposait un choc écrit à l'avance.

Le choc est un paramètre de `src/config.py`. Il n'est pas caché dans un scénario nommé « 2008 » ou « Covid » dont les dates seraient figées dans le code. Si la période est prolongée d'un mois, le choc reste le même énoncé, appliqué aux données du moment.

Le choc de récession remplace les mêmes mois par le même rendement dans les trois livres. Il ne désigne pas le titre qui souffre le plus. Cette question resterait une question de composition. Elle est traitée par les poids de la phase 6 et par les corrélations de la phase 8.

## Où en est le calcul

Le module est `src/stress_testing/build_stress.py`. La commande est `python -m src.stress_testing.build_stress`. Le rafraîchissement l'appelle après le risque. Les poids ne sont pas réestimés. La ligne sans choc doit retrouver la phase 5 et la phase 6, à \(10^{-8}\) près. Sinon le calcul s'arrête.


## Sources lues

On part du rendement mensuel déjà publié. On ne rejoue pas l'optimiseur : les poids restent ceux de la phase 5. Un choc qui changerait les poids serait un autre portefeuille, pas un test sur celui-ci.

| Fichier | Origine | Usage |
|---|---|---|
| `data/04_portfolios/portfolio_monthly_returns.csv` | phase 5 | La série à laquelle on applique le choc de marché, le choc de récession et le choc de volatilité |
| `data/04_portfolios/portfolio_weights.csv` | phase 5 | Contrôle : les poids de chaque stratégie somment à 1. Sans cela, un choc identique sur chaque titre ne se reporte pas tel quel sur le portefeuille |
| `data/02_processed/macro/saron.csv` | `DS-005` | Le choc de taux reconstruit le rendement sans risque à partir du SARON décalé |

`DS-003` et `DS-004` ne sont pas relus. Leur effet est déjà dans `portfolio_return`. `DS-000` et `DS-006` ne sont pas lus. Un statut ODD Indeterminate ne définit pas un scénario.

Les quatre chocs utilisent les valeurs suivantes. Elles devront vivre dans `src/config.py` quand le module sera écrit, pas dans une cellule isolée.

| Choc | Paramètre | Valeur retenue |
|---|---|---|
| Marché | Rendement imposé au dernier mois | $-20\%$ |
| Taux | Décalage du SARON annuel, toute la période | $+2$ points de pourcentage |
| Récession | Rendement imposé à chacun des six derniers mois | $-3\%$ par mois |
| Volatilité | Multiplicateur des écarts à la moyenne | $1{,}5$ |


## Choc de marché

On remplace le rendement du dernier mois par $-20\%$. Les mois précédents restent les rendements publiés. Comme les poids somment à 1 et que le portefeuille est en actions, un choc de $-20\%$ sur chaque titre le même mois est le même chiffre que ce remplacement : $\sum_i w_i (R_{i,T} - 0{,}20) = R_{p,T} - 0{,}20$, puis on impose quand même $-20\%$, pas $R_{p,T} - 0{,}20$. Le paramètre est le rendement du mois, pas un complément au rendement observé.

On recalcule ensuite la performance cumulée et le drawdown maximal avec les formules de la phase 6. On les compare à la série non choquée.


## Choc de taux

Le SARON du fichier est un pourcentage annuel. Le choc lui ajoute 2 points sur chaque mois de la période. Un SARON de $1{,}5$ devient $3{,}5$. On ne touche pas aux rendements des titres.

Le rendement sans risque du mois redevient

$$
r_{f,t}^{\star} = \left(1 + \frac{\mathrm{SARON}_t + 2}{100}\right)^{1/12} - 1
$$

L'excédent choqué est $E_{p,t}^{\star} = R_{p,t} - r_{f,t}^{\star}$. Le Sharpe est recalculé avec la formule de la phase 5, sur ces excédents. Les poids ne changent pas. Un SARON plus haut baisse l'excédent. Il ne baisse pas le rendement du portefeuille.


## Choc de récession

On remplace les six derniers mois par un rendement de $-3\%$ chacun. Le reste de la série reste publié. Six mois à $-3\%$ multiplient la performance cumulée par $0{,}97^6$ sur ce tronçon, quel que soit le portefeuille : le choc ne dépend pas des poids. Ce choix est volontaire. Il montre la sensibilité du drawdown maximal à une séquence défavorable identique, pas une différence de composition.

La performance cumulée et le drawdown maximal sont recalculés sur la série modifiée.


## Choc de volatilité

On garde la moyenne $\bar R_p$ de la série. Chaque mois est éloigné de cette moyenne :

$$
R_{p,t}^{\star} = \bar R_p + 1{,}5 \, (R_{p,t} - \bar R_p)
$$

La moyenne ne change pas. L'écart-type est multiplié par $1{,}5$, aux arrondis près. Le Sharpe, lui, change, parce que l'excédent $R_{p,t}^{\star} - r_{f,t}$ n'a plus la même dispersion, et parce que le taux sans risque n'est pas multiplié. On publie la volatilité choquée et le Sharpe choqué, à côté des valeurs d'origine.


## Fichiers publiés

`data/08_stress/stress_results.csv`. Une ligne par stratégie et par choc, plus une ligne `aucun` qui répète la série d'origine.

| Colonne | Sens |
|---|---|
| `strategy` | Une des trois stratégies |
| `shock` | `aucun`, `marche`, `taux`, `recession`, `volatilite` |
| `months`, `window_start`, `window_end` | La fenêtre déjà publiée |
| `terminal_wealth` | Performance cumulée en fin de série choquée |
| `maximum_drawdown` | Drawdown maximal de cette série. Il est négatif |
| `annualized_volatility` | Écart-type annualisé de la série choquée. Le choc de volatilité doit le multiplier par le paramètre |
| `sharpe_ratio` | Sharpe de cette série, formule de la phase 5. Pour le marché et la récession, le taux sans risque reste celui du fichier. Pour le taux, il est reconstruit avec le SARON décalé |

`data/08_stress/stress_paths.csv` conserve, pour chaque mois choqué, le rendement, le taux sans risque, la performance cumulée et le drawdown. La page lit ce chemin. Elle ne réapplique pas le choc.

La cellule suivante relit ces fichiers. Elle s'arrête si un choc ne retrouve pas son paramètre.


In [1]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import (
    MARKET_SHOCK_RETURN,
    RATE_SHOCK_ANNUAL_POINTS,
    RECESSION_SHOCK_MONTHS,
    RECESSION_SHOCK_RETURN,
    VOLATILITY_SHOCK_MULTIPLIER,
)
from src.stress_testing.build_stress import AGREEMENT, PATHS_FILE, RESULTS_FILE

LABELS = {
    "equal_weight": "Équipondéré",
    "minimum_volatility": "Volatilité minimale",
    "maximum_sharpe": "Sharpe maximal",
}
SHOCKS = {
    "aucun": "Sans choc",
    "marche": "Marché",
    "taux": "Taux",
    "recession": "Récession",
    "volatilite": "Volatilité",
}


def percent(value):
    return f"{float(value) * 100:.1f}".replace(".", ",") + " %"


results = pd.read_csv(RESULTS_FILE)
paths = pd.read_csv(PATHS_FILE)
print(
    f"Marché : dernier mois remplacé par {percent(MARKET_SHOCK_RETURN)}. "
    f"Taux : SARON annuel décalé de {RATE_SHOCK_ANNUAL_POINTS:.0f} points. "
    f"Récession : {RECESSION_SHOCK_MONTHS} mois à {percent(RECESSION_SHOCK_RETURN)}. "
    f"Volatilité : écarts multipliés par {VOLATILITY_SHOCK_MULTIPLIER:.1f}."
)
start = results["window_start"].iloc[0]
end = results["window_end"].iloc[0]
print(f"Période {start} – {end}, {int(results['months'].iloc[0])} mois.")
for strategy, label in LABELS.items():
    print(label)
    block = results.loc[results["strategy"].eq(strategy)].set_index("shock")
    base = block.loc["aucun"]
    for shock, shock_label in SHOCKS.items():
        row = block.loc[shock]
        print(
            f"  {shock_label} : cumulée {float(row['terminal_wealth']):.2f}, "
            f"drawdown {percent(row['maximum_drawdown'])}, "
            f"volatilité {percent(row['annualized_volatility'])}, "
            f"Sharpe {float(row['sharpe_ratio']):.2f}."
        )
    if abs(float(block.loc["taux", "terminal_wealth"]) - float(base["terminal_wealth"])) > AGREEMENT:
        raise RuntimeError(f"{strategy} rate shock changed terminal wealth.")
    ratio = float(block.loc["volatilite", "annualized_volatility"]) / float(base["annualized_volatility"])
    if abs(ratio - VOLATILITY_SHOCK_MULTIPLIER) > AGREEMENT:
        raise RuntimeError(f"{strategy} volatility shock did not scale.")
    trail = paths.loc[paths["strategy"].eq(strategy)].sort_values("month")
    market = trail.loc[trail["shock"].eq("marche"), "portfolio_return"].iloc[-1]
    if abs(float(market) - MARKET_SHOCK_RETURN) > AGREEMENT:
        raise RuntimeError(f"{strategy} market shock did not replace the last month.")
    recession = trail.loc[trail["shock"].eq("recession"), "portfolio_return"].iloc[-RECESSION_SHOCK_MONTHS :]
    if float(np.max(np.abs(recession.to_numpy(dtype=float) - RECESSION_SHOCK_RETURN))) > AGREEMENT:
        raise RuntimeError(f"{strategy} recession shock did not replace the last months.")
print("Les trois livres retrouvent les paramètres. La récession impose le même rendement. Elle ne désigne pas un titre.")


Marché : dernier mois remplacé par -20,0 %. Taux : SARON annuel décalé de 2 points. Récession : 6 mois à -3,0 %. Volatilité : écarts multipliés par 1.5.
Période 2016-11 – 2026-10, 120 mois.
Équipondéré
  Sans choc : cumulée 2.15, drawdown -22,9 %, volatilité 12,3 %, Sharpe 0.69.
  Marché : cumulée 1.68, drawdown -22,9 %, volatilité 13,9 %, Sharpe 0.45.
  Taux : cumulée 2.15, drawdown -22,9 %, volatilité 12,3 %, Sharpe 0.53.
  Récession : cumulée 1.64, drawdown -23,7 %, volatilité 12,5 %, Sharpe 0.46.
  Volatilité : cumulée 1.96, drawdown -39,1 %, volatilité 18,4 %, Sharpe 0.46.
Volatilité minimale
  Sans choc : cumulée 1.84, drawdown -23,1 %, volatilité 11,8 %, Sharpe 0.59.
  Marché : cumulée 1.47, drawdown -25,0 %, volatilité 13,4 %, Sharpe 0.36.
  Taux : cumulée 1.84, drawdown -23,1 %, volatilité 11,8 %, Sharpe 0.42.
  Récession : cumulée 1.45, drawdown -26,0 %, volatilité 12,0 %, Sharpe 0.38.
  Volatilité : cumulée 1.69, drawdown -38,3 %, volatilité 17,6 %, Sharpe 0.39.
Sharpe maxim

# Dictionnaire des termes de cette note

Ce dictionnaire reprend les termes employés dans cette note. Il n'en ajoute pas. Les formules restent dans la section qui les écrit.

**Choc de marché.** Le rendement du dernier mois est remplacé par −20 %. Les mois précédents restent publiés.

**Choc de récession.** Chacun des six derniers mois est remplacé par −3 %.

**Choc de taux.** Le SARON annuel de chaque mois est relevé de 2 points de pourcentage. Les rendements des titres ne changent pas.

**Choc de volatilité.** Chaque rendement mensuel est éloigné de la moyenne par un facteur 1,5. La moyenne reste la même.

**Drawdown maximal.** Plus forte baisse de la performance cumulée, recalculée sur la série choquée.

**Indeterminate.** Statut ODD non lu. Il ne définit pas un scénario.

**Performance cumulée.** Valeur d'un placement qui enchaîne les rendements, d'origine ou choqués.

**Ratio de Sharpe.** Rendement au-dessus du taux sans risque, divisé par la volatilité. Il est recalculé sur la série choquée avec la formule de la phase 5.

**SARON.** Taux sans risque annuel. Le choc de taux le décale. Le rendement mensuel sans risque est reconstruit à partir de ce taux décalé.

**Test de résistance.** Question posée avec un choc écrit à l'avance. Les poids restent ceux de la phase 5.
